# AlphaTrain Pillar 3H: the 2z recipe, on V15 data

**The point of this run:** apply the *proven* 2z/2y generational-distillation recipe to the V15 self-play corpus. We never actually did this. pillar3g deviated (`train_path_b` + `--target-temperature 0.5` sharpening) and regressed; pillar3g2 deviated harder (completed-Q) and regressed worse. This run is **plain, unsharpened policy distillation** — `alphatrain.train --policy-only`, `target_temperature=1.0` (the V12/2z default), lr 3e-4, warmup 2, bs 32768 — identical to the command that produced pillar2z (MCTS mean 15,465, a new best at the time).

**Data:** `v15_pillar3f_slim.pt` (5.35M states; relabel_v15 clean@400 + selfplay_v15). Built with `build_expert_v2_tensor.py --policy-only-data`, so it carries `pol_indices/pol_values` (the visit policy target) exactly like the 2z tensor. The completed-Q fields it also stores are simply ignored by `alphatrain.train`.

**Model:** warm-start from pillar3f (10 blocks × 256ch).

**Watch-point:** pillar3f is a much stronger base than 2z's (raw policy ~23k vs 2z's 7.5k), so the per-generation gain may be smaller than the 2z era — but this is the proven recipe and it has never been run on V15.

**Eval (the success metric):** evaluate WITH MCTS@400 + value_head_pillar3f (2z's win was an MCTS-player win, not a raw-policy win), as a **distribution over a seed range** (P5/P10/floor + mean), vs pillar3f — never per-seed.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_pillar3d_v2.tar.gz` — code (current; includes `alphatrain/train.py`)
2. `v15_pillar3f_slim.pt.gz` — the V15 slim tensor (gzip of the ~2 GB file)
3. `pillar3f.pt` — warm-start weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3f.pt', '/content/alphatrain/data/pillar3f.pt')
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v15_pillar3f_slim.pt.gz > /content/alphatrain/data/v15_pillar3f_slim.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/v15_pillar3f_slim.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# sanity: this tensor MUST carry the policy target (pol_indices/pol_values) that
# alphatrain.train --policy-only distills, exactly like the 2z tensor.
import torch
d = torch.load('/content/alphatrain/data/v15_pillar3f_slim.pt', weights_only=True)
assert 'pol_indices' in d and 'pol_values' in d, 'tensor missing policy target!'
print(f"states: {d['boards'].shape[0]:,} | pol target present ✓ | value_mode={d.get('value_mode')}")
del d

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
!cd /content && python -m pytest alphatrain/tests/test_model.py -q --tb=short 2>&1 | tail -4

In [ ]:
# Pillar 3H = the 2z recipe, verbatim, on V15. PLAIN soft-CE policy distillation:
# NO --target-temperature (defaults to 1.0 = unsharpened, the V12/2z baseline),
# alphatrain.train (NOT train_path_b), --policy-only, lr 3e-4, warmup 2, bs 32768.
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/v15_pillar3f_slim.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --policy-only \
    --epochs 20 --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3h_best.pt \
    --save-dir /content/checkpoints/pillar3h
# If val_loss is still falling at epoch 20, extend with --resume (2z went to 40).

In [ ]:
# Copy ALL epoch checkpoints to Drive (eval them WITH MCTS@400 + value_head_pillar3f,
# distribution over a seed range, vs pillar3f — the 2z success metric).
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar3h/epoch_*.pt')):
    dst = f'{DRIVE}/pillar3h_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar3h/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar3h_{f}')